# 🎗️ Cancer Treatment Cost Prediction

**Predicting the Treatment Cost (USD) of cancer patients using 6 regression models:**
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- K-Nearest Neighbors Regressor
- Support Vector Regressor (SVR)
- Gradient Boosting Regressor

**Target column:** `Treatment_Cost_USD`  
**Dataset:** Global Cancer Patients Dataset

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (10, 5)

print('All libraries loaded successfully!')

## 2. Load & Explore Dataset

In [ ]:
# Kaggle datasets are mounted at /kaggle/input/
# Update the path below to match your dataset slug on Kaggle
DATA_PATH = '/kaggle/input/cancer-prediction-dataset/cancer.csv'

df_raw = pd.read_csv(DATA_PATH)
print('Shape:', df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe()

In [ ]:
print('Missing values per column:')
print(df_raw.isnull().sum())

## 3. Preprocessing

In [ ]:
df = df_raw.copy()

# Drop null records
df = df.dropna()

# Drop unwanted columns
df = df.drop(['Patient_ID', 'Survival_Years'], axis=1)

# Label Encoding for categorical columns
encoder = LabelEncoder()
categorical_cols = ['Gender', 'Country_Region', 'Cancer_Type', 'Cancer_Stage']

for col in categorical_cols:
    df[col] = encoder.fit_transform(df[col])

print('Preprocessed shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

In [ ]:
# Target distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(df['Treatment_Cost_USD'], bins=40, color='steelblue', edgecolor='white')
plt.title('Treatment Cost Distribution')
plt.xlabel('Treatment Cost (USD)')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
df['Cancer_Type'].value_counts().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Cancer Type Frequency')
plt.xlabel('Cancer Type (encoded)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Train / Test Split

In [ ]:
X = df.drop('Treatment_Cost_USD', axis=1)
y = df['Treatment_Cost_USD']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=10
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')

## 5. Model Training & Evaluation

A helper function evaluates every trained model and stores the results.

In [ ]:
results = {}

def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    r2  = r2_score(y_te, y_pred)
    mse = mean_squared_error(y_te, y_pred)
    mae = mean_absolute_error(y_te, y_pred)
    results[name] = {'R2 Score': round(r2 * 100, 4),
                     'MSE': round(mse, 4),
                     'MAE': round(mae, 4)}
    print(f'[{name}]')
    print(f'  R2 Score (accuracy) : {r2 * 100:.4f} %')
    print(f'  MSE                 : {mse:.4f}')
    print(f'  MAE                 : {mae:.4f}')
    print()

    # Save model
    os.makedirs(f'/kaggle/working/{name}', exist_ok=True)
    joblib.dump(model, f'/kaggle/working/{name}/train_model.dat')
    return y_pred

### 5.1 Linear Regression

In [ ]:
lr_pred = evaluate('LinearRegression',
                   LinearRegression(),
                   X_train, y_train, X_test, y_test)

### 5.2 Decision Tree Regressor

In [ ]:
dt_pred = evaluate('DecisionTree',
                   DecisionTreeRegressor(max_depth=15, random_state=42),
                   X_train, y_train, X_test, y_test)

### 5.3 Random Forest Regressor

In [ ]:
rf_pred = evaluate('RandomForest',
                   RandomForestRegressor(max_depth=15, random_state=42, n_estimators=100),
                   X_train, y_train, X_test, y_test)

### 5.4 K-Nearest Neighbors Regressor

In [ ]:
knn_pred = evaluate('KNN',
                    KNeighborsRegressor(),
                    X_train, y_train, X_test, y_test)

### 5.5 Support Vector Regressor (SVR)

In [ ]:
svr_pred = evaluate('SVR',
                    SVR(kernel='rbf'),
                    X_train, y_train, X_test, y_test)

### 5.6 Gradient Boosting Regressor

In [ ]:
gb_pred = evaluate('GradientBoosting',
                   GradientBoostingRegressor(max_depth=15, random_state=42),
                   X_train, y_train, X_test, y_test)

## 6. Results Comparison

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('R2 Score', ascending=False)
results_df

In [ ]:
# Bar chart — R2 Score comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

colors = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6', '#ec4899']
models = results_df.index.tolist()

axes[0].barh(models, results_df['R2 Score'], color=colors)
axes[0].set_title('R2 Score (%)')
axes[0].set_xlabel('R2 Score')

axes[1].barh(models, results_df['MSE'], color=colors)
axes[1].set_title('Mean Squared Error')
axes[1].set_xlabel('MSE')

axes[2].barh(models, results_df['MAE'], color=colors)
axes[2].set_title('Mean Absolute Error')
axes[2].set_xlabel('MAE')

plt.tight_layout()
plt.savefig('/kaggle/working/model_comparison.png', dpi=150)
plt.show()
print('Saved: /kaggle/working/model_comparison.png')

In [ ]:
# Actual vs Predicted — best model
best_model_name = results_df.index[0]
best_model = joblib.load(f'/kaggle/working/{best_model_name}/train_model.dat')
best_pred  = best_model.predict(X_test)

plt.figure(figsize=(8, 5))
plt.scatter(y_test, best_pred, alpha=0.4, color='steelblue', s=15)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect fit')
plt.xlabel('Actual Treatment Cost (USD)')
plt.ylabel('Predicted Treatment Cost (USD)')
plt.title(f'Actual vs Predicted — {best_model_name}')
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/actual_vs_predicted.png', dpi=150)
plt.show()
print(f'Best Model: {best_model_name}  |  R2: {results_df.loc[best_model_name, "R2 Score"]}%')

## 7. Feature Importance (Random Forest)

In [ ]:
rf_model = joblib.load('/kaggle/working/RandomForest/train_model.dat')
importances = pd.Series(rf_model.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Feature Importances — Random Forest')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('/kaggle/working/feature_importance.png', dpi=150)
plt.show()

## 8. Conclusion

| Model | R2 Score | Notes |
|---|---|---|
| Linear Regression | baseline | Fast, interpretable |
| Decision Tree | depth=15 | Prone to overfit |
| Random Forest | depth=15, 100 trees | Robust ensemble |
| KNN | default k=5 | Distance-based |
| SVR | RBF kernel | Non-linear mapping |
| Gradient Boosting | depth=15 | Boosted ensemble |

All trained models are saved to `/kaggle/working/<ModelName>/train_model.dat` and can be downloaded from the **Output** tab.